# Target 10m Model Experiments Overview

Обзор результатов staged hold-out перебора моделей для `target_log_return_10m`.

Локальные входные файлы скачаны из:

`s3://binance-data-downloader/dataset_target_10/model_experiments/latest/`

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 180)
plt.style.use("seaborn-v0_8-whitegrid")

RESULTS_DIR = Path("analysis/target_10_model_experiments/results/latest")
if not RESULTS_DIR.exists():
    RESULTS_DIR = Path("results/latest")

results = pd.read_parquet(RESULTS_DIR / "experiment_results.parquet")
stage_selection = pd.read_parquet(RESULTS_DIR / "stage_selection.parquet")
run_config = json.loads((RESULTS_DIR / "run_config.json").read_text(encoding="utf-8"))

results.shape, stage_selection.shape

## Run Summary

In [ ]:
summary_keys = [
    "run_id",
    "target",
    "direction_threshold",
    "train_rows",
    "test_rows",
    "n_jobs",
    "parallel_backend",
    "max_selected_per_stage",
]
pd.DataFrame(
    [{"field": key, "value": run_config.get(key)} for key in summary_keys]
)

In [ ]:
stage_selection

## Global Top Models

In [ ]:
display_cols = [
    "model_id",
    "stage",
    "model_family",
    "feature_set",
    "n_features",
    "MAE",
    "RMSE",
    "Direction_Accuracy",
    "Direction_Accuracy_025",
    "DA_025_coverage",
    "OOS_R2",
    "Pearson_corr",
]

top_a = results.sort_values(
    ["MAE", "RMSE", "n_features", "simplicity_rank", "model_id"],
    ascending=[True, True, True, True, True],
).head(15)

top_b = results.sort_values(
    ["Direction_Accuracy_025", "n_features", "simplicity_rank", "model_id"],
    ascending=[False, True, True, True],
    na_position="last",
).head(15)

top_a[display_cols]

In [ ]:
top_b[display_cols]

## Best Model By Stage

In [ ]:
best_mae_by_stage = (
    results.sort_values(["stage", "MAE", "RMSE", "n_features", "simplicity_rank"])
    .groupby("stage", as_index=False)
    .first()
)

best_da_by_stage = (
    results.sort_values(
        ["stage", "Direction_Accuracy_025", "n_features", "simplicity_rank"],
        ascending=[True, False, True, True],
        na_position="last",
    )
    .groupby("stage", as_index=False)
    .first()
)

best_mae_by_stage[display_cols]

In [ ]:
best_da_by_stage[display_cols]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

best_mae_by_stage.plot(x="stage", y="MAE", marker="o", ax=axes[0], legend=False)
axes[0].set_title("Best MAE By Stage")
axes[0].tick_params(axis="x", rotation=45)

best_mae_by_stage.plot(x="stage", y="RMSE", marker="o", ax=axes[1], legend=False, color="tab:orange")
axes[1].set_title("Best RMSE By Stage")
axes[1].tick_params(axis="x", rotation=45)

best_da_by_stage.plot(x="stage", y="Direction_Accuracy_025", marker="o", ax=axes[2], legend=False, color="tab:green")
axes[2].set_title("Best DA >= 0.25% By Stage")
axes[2].tick_params(axis="x", rotation=45)

plt.tight_layout()

## Added Block Informativeness

In [ ]:
block_summary = (
    results.loc[results["stage"].ne("stage_0_baseline")]
    .groupby(["stage", "added_block"], as_index=False)
    .agg(
        models=("model_id", "count"),
        best_MAE=("MAE", "min"),
        median_MAE=("MAE", "median"),
        best_RMSE=("RMSE", "min"),
        best_DA_025=("Direction_Accuracy_025", "max"),
        median_DA_025=("Direction_Accuracy_025", "median"),
        best_OOS_R2=("OOS_R2", "max"),
        median_delta_MAE=("delta_MAE", "median"),
        median_delta_RMSE=("delta_RMSE", "median"),
        median_delta_DA_025=("delta_DA_025", "median"),
    )
    .sort_values(["stage", "best_MAE", "best_RMSE"])
)

block_summary

In [ ]:
for stage, frame in block_summary.groupby("stage"):
    top_blocks = frame.sort_values("best_MAE").head(8)
    ax = top_blocks.plot.barh(
        x="added_block",
        y="best_MAE",
        figsize=(9, max(3, len(top_blocks) * 0.45)),
        legend=False,
        title=f"{stage}: best MAE by added block",
    )
    ax.invert_yaxis()
    plt.show()

## Selected Models By Stage

In [ ]:
selected = results.loc[results["selected_for_next_stage"]].copy()
selected.sort_values(["stage", "rank_A", "rank_B"])[display_cols + ["rank_A", "rank_B", "selection_reason", "previous_model", "added_block"]]

## Model Family Diagnostics

In [ ]:
family_summary = (
    results.groupby(["stage", "model_family"], as_index=False)
    .agg(
        models=("model_id", "count"),
        best_MAE=("MAE", "min"),
        median_MAE=("MAE", "median"),
        best_DA_025=("Direction_Accuracy_025", "max"),
        median_DA_025=("Direction_Accuracy_025", "median"),
        best_OOS_R2=("OOS_R2", "max"),
        total_fit_time=("fit_time", "sum"),
    )
    .sort_values(["stage", "best_MAE"])
)

family_summary

## Notes For Interpretation

- `Direction_Accuracy_025` считается только на наблюдениях, где `abs(y_true) >= 0.25%`.
- `OOS_R2` считается относительно zero baseline.
- `delta_*` считается относительно `previous_model`, то есть показывает локальный эффект добавленного блока внутри цепочки отбора.
- Финальные победители по MAE и directional accuracy могут отличаться: это ожидаемо для задачи точечной доходности.